In [1]:
import os 

In [2]:
%pwd

'c:\\Users\\Asus\\Machine_learning\\LLM\\Language_Model\\GPT2_124M\\notebook'

In [3]:
os.chdir("..//")

In [4]:
os.chdir("..//")

In [5]:
import math 
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
# @title Model

class CasualSelfAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        assert config.n_emb % config.n_head ==0
        self.c_attn = nn.Linear(config.n_emb,3*config.n_emb)    # combined attention==> key,query,value projection for all heads,but in a batch
        self.c_proj = nn.Linear(config.n_emb,config.n_emb)      # output projection
        #self.c_proj.NANOGPT_SCALE_INIT  = 1

        self.n_head = config.n_head
        self.n_emb  = config.n_emb
        self.register_buffer("bias",torch.tril(torch.ones(config.block_size,config.block_size)).view(1,1,config.block_size,config.block_size))

    def forward(self,x):
        B,T,C       = x.shape # batch_size, sequence length, embedding dim
        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        # nh                    = "number of heads",
        # hs                    = "head size"
        # C (number of channels)= nh * hs
        # e.g. in GPT-2 (124M)==> n_head    =12,
        #                         hs        =64, ==> nh * hs = C = 768 channels in the Transformer
        qkv     = self.c_attn(x)                                                    # B,T,3*n_emb
        q,k,v   = qkv.split(self.n_emb,dim=2)                                       # B,T,n_emb
        q       = q.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        k       = k.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        v       = v.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        ## Attention 
        atten   = (q@k.transpose(-2,-1)) * (1.0/math.sqrt(k.size(-1)))
        atten   = atten.masked_fill(self.bias[:,:,:T,:T] == 0,float("-inf"))
        atten   = F.softmax(atten,dim=-1)

        y       = atten @ v                 # (B,nh,T,T) x (B,nh,T,hs) ==> (B,nh,T,hs)
        y       = y.transpose(1,2).contiguous().view(B,T,C)
        #output projection
        y       = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_emb,4*config.n_emb)
        self.gelu   = nn.GELU(approximate="tanh")   # there is no reason to use this approximation in nowdays, the time they develop this approximation they faced speed issue. thats why developed approximation
        self.c_proj = nn.Linear(config.n_emb * 4,config.n_emb)
        #self.c_proj.NANOGPT_SCALE_INIT  = 1
    def forward(self,x):
        x   = self.c_fc(x)
        x   = self.gelu(x)
        x   = self.c_proj(x)
        return x
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1   = nn.LayerNorm(config.n_emb)
        self.attn   = CasualSelfAttention(config)
        self.ln_2   = nn.LayerNorm(config.n_emb)
        self.mlp    = MLP(config)

    def forward(self,x):
        x   = x + self.attn(self.ln_1(x))
        x   = x + self.mlp(self.ln_2(x))
        return x


In [8]:
@dataclass
class GPTConfig:
    block_size:int  = 1024  # ==> block size
    vocab_size:int  = 50257 # ==> numbers of tokens ==> 50000 merges + 256 bytes token + 1 special token <|endoftext|>
    n_layer:int     = 12
    n_head:int      = 12
    n_emb:int       = 768   # ==> embedding dim

In [9]:
class GPT(nn.Module):
  def __init__(self,config):
      super().__init__()
      self.config = config
      self.transformer = nn.ModuleDict(dict(
          wte     = nn.Embedding(config.vocab_size,config.n_emb),           # token embeding
          wpe     = nn.Embedding(config.block_size,config.n_emb),           # position embedding
          h       = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),    # self attention heads
          ln_f    = nn.LayerNorm(config.n_emb)
      ))
      self.lm_head= nn.Linear(config.n_emb,config.vocab_size,bias=False)    # lm_head is following be softmax, and bias not make any sence or improvement in learning.
      # The bias term in this case would just add a constant to each token’s logit — this doesn’t meaningfully improve learning,

  def forward(self,idx,target=None):
    # shape of idx is (B,T)
    B,T     = idx.shape
    assert T<=self.config.block_size, f"cannot forward sequence of length {T},block_size is only {self.config.block_size}"
    pos     = torch.arange(0,T,dtype=torch.long,device=idx.device)  # shape (T)
    pos_emb = self.transformer.wpe(pos)                             # position embedding of shape (_,T,n_emb)
    tok_emb = self.transformer.wte(idx)                             # token embedding of shape    (B,T,n_emb)

    x       = tok_emb + pos_emb         # (B,T,n_emb)
    for block in self.transformer.h:
      x = block(x)
    #forward the final layerorm and classifier
    x       = self.transformer.ln_f(x)
    logits  = self.lm_head(x)           # (B,T,n_emb)

    ##------------------------------Adding Target and Loss---------------------- ##
    loss    = None
    if target is None:
      loss  = None
    elif target is not None:
      loss  = F.cross_entropy(input   = logits.view(-1,logits.size(-1)),        # cross entropy does not like multi-dimensional input, flatten out into 2D
                              target  = target.view(-1),)
    return logits,loss

In [10]:
import time
import tiktoken

## Data Loading

In [11]:
enc         = tiktoken.get_encoding("gpt2")
text_path   = "data\gpt_train.txt"
with open(text_path,"r") as f:
   text = f.read() 
data = text[:1000]

tokens    = enc.encode(data)

In [12]:
sample  = tokens[:24 + 1] # Take 25 elements: 24 for the input sequence (X) and 1 for the target (y) matrix.
x       = torch.tensor(sample[:-1]).view(4,6).to(device)
y       = torch.tensor(sample[1:]).view(4,6).to(device)

In [13]:
class DataLoaderLite:
    def __init__(self,B,T):
        self.B  = B     # batch 
        self.T  = T     # sequence length 
        with open("data\gpt_train.txt","r") as f:
            text    = f.read()
        enc         = tiktoken.get_encoding("gpt2")
        tokens      = enc.encode(text)
        self.tokens = torch.tensor(tokens)
        print(f"loaded of {len(self.tokens)} tokens")
        print(f"1 Epoch = {len(self.tokens) // (B*T)} Batches of token")

        self.current_position = 0 

    def next_batch(self):
        B,T     = self.B,self.T
        buff    = self.tokens[self.current_position:self.current_position+B*T+1]
        x       = (buff[:-1]).view(B,T)     # input 
        y       = (buff[1:]).view(B,T)      # target 
        self.current_position   += B*T 
        if self.current_position + (B*T+1)> len(self.tokens):
            self.current_position = 0 
        return x,y

In [14]:
train_loader = DataLoaderLite(B=4,T=32)  

loaded of 338025 tokens
1 Epoch = 2640 Batches of token


## Model Training 

In [15]:
model = GPT(GPTConfig())
model.eval()
model = model.to(device)

#### Initial loss expecting = -ln(1/50257) = 10.82 

In [16]:
## Optimizer 
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)
for i in range(50):
    x,y = train_loader.next_batch()
    x,y = x.to(device),y.to(device)
    t0  = time.time()
    optimizer.zero_grad()
    logits,loss = model(x,y)
    loss.backward()
    optimizer.step()
    t1  = time.time()
    dt = (t1 - t0) * 1000  # convert seconds to milliseconds
    print(f"Step : {i} loss: {loss.item():.4f}, dt: {dt:.4f} ms")

c:\Users\Asus\anaconda3\envs\llm_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Step : 0 loss: 11.0525, dt: 549.8624 ms
Step : 1 loss: 9.6848, dt: 38.3260 ms
Step : 2 loss: 8.5915, dt: 20.4306 ms
Step : 3 loss: 8.9287, dt: 31.7492 ms
Step : 4 loss: 8.4729, dt: 25.5675 ms
Step : 5 loss: 8.1492, dt: 28.1436 ms
Step : 6 loss: 8.9288, dt: 16.3331 ms
Step : 7 loss: 8.7947, dt: 34.7819 ms
Step : 8 loss: 7.9593, dt: 25.0642 ms
Step : 9 loss: 7.8440, dt: 27.3848 ms
Step : 10 loss: 8.2694, dt: 28.3930 ms
Step : 11 loss: 7.1980, dt: 31.7452 ms
Step : 12 loss: 7.6471, dt: 27.0782 ms
Step : 13 loss: 7.3565, dt: 26.5279 ms
Step : 14 loss: 7.4056, dt: 21.4877 ms
Step : 15 loss: 7.1446, dt: 28.5511 ms
Step : 16 loss: 7.2751, dt: 31.8367 ms
Step : 17 loss: 8.3291, dt: 33.8602 ms
Step : 18 loss: 7.0882, dt: 28.0008 ms
Step : 19 loss: 7.7312, dt: 33.8430 ms
Step : 20 loss: 7.3927, dt: 29.1934 ms
Step : 21 loss: 7.5376, dt: 25.7218 ms
Step : 22 loss: 6.1906, dt: 26.3858 ms
Step : 23 loss: 6.5514, dt: 30.2420 ms
Step : 24 loss: 6.5603, dt: 26.7065 ms
Step : 25 loss: 6.3013, dt: 29.66

- In initial stage im expecting loss is coming down but not too much, Because from token=50257, many of those tokens never occurs in this dataset that we used to train the model. 
- So there is easy gain made here in this optimization. 
    - Bias of logits never occurs driving them to -infinity. (Because the 50257 form those tokens that never occurs in the dataset,so the probability of those tokens really low )
    - **Easy Optimization Gains**: This early reduction is primarily attributed to "easy gains" in optimization.

- **Bias Towards Negative Infinity**: The absence of these tokens causes their corresponding logit biases to approach negative infinity, effectively driving their predicted probabilities to near zero. This correction contributes significantly to the initial loss reduction.

## DeBugging 

- If you check the matric of `transformer.wte.weight` and `lm_head.weight`of gpt2 model, both matrix are similar and point to a single **data_pointer**. 
- Research paper:- `https://arxiv.org/pdf/1608.05859`

- `st_dict = model.state_dict()`
- `st_dict['transformer.wte.weight] == st_dict['lm_head.weight]` 
& 
- `st_dict['transformer.wte.weight].data_ptr() == st_dict[lm_head.weight].data_ptr()`  

- Using same weight for input and output embedding improves language modeling. 

In [30]:
class GPT(nn.Module):
  def __init__(self,config):
      super().__init__()
      self.config = config
      self.transformer = nn.ModuleDict(dict(
          wte     = nn.Embedding(config.vocab_size,config.n_emb),           # token embeding
          wpe     = nn.Embedding(config.block_size,config.n_emb),           # position embedding
          h       = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),    # self attention heads
          ln_f    = nn.LayerNorm(config.n_emb)
      ))
      self.lm_head= nn.Linear(config.n_emb,config.vocab_size,bias=False)    # lm_head is following be softmax, and bias not make any sence or improvement in learning.
      # The bias term in this case would just add a constant to each token’s logit — this doesn’t meaningfully improve learning,
      
      #-----------------------weight sharing scheme---------------------------------# 
      self.transformer.wte.weight  = self.lm_head.weight

  def forward(self,idx,target=None):
    # shape of idx is (B,T)
    B,T     = idx.shape
    assert T<=self.config.block_size, f"cannot forward sequence of length {T},block_size is only {self.config.block_size}"
    pos     = torch.arange(0,T,dtype=torch.long,device=idx.device)  # shape (T)
    pos_emb = self.transformer.wpe(pos)                             # position embedding of shape (_,T,n_emb)
    tok_emb = self.transformer.wte(idx)                             # token embedding of shape    (B,T,n_emb)

    x       = tok_emb + pos_emb         # (B,T,n_emb)
    for block in self.transformer.h:
      x = block(x)
    #forward the final layerorm and classifier
    x       = self.transformer.ln_f(x)
    logits  = self.lm_head(x)           # (B,T,n_emb)

    ##------------------------------Adding Target and Loss---------------------- ##
    loss    = None
    if target is None:
      loss  = None
    elif target is not None:
      loss  = F.cross_entropy(input   = logits.view(-1,logits.size(-1)),        # cross entropy does not like multi-dimensional input, flatten out into 2D
                              target  = target.view(-1),)
    return logits,loss

- By doing sharing weights between embedding(`transformer.wte.weight`) and lm_head (`lm_head.weight`), we dont have to train 2 seperate large matrices of parameters, `n_emb * vocab_size = 768 * 50257 closer to 40 million`. 
- The Model is 124 Million parameter model. 40Million is almost 30% of total parameters. 
### ------------------------------------------------------------------------------------------------------------ ###
- 1. Improves Training Stability and Accuracy
    - If a word is embedded a certain way on input, it should be decoded similarly on output.
    - This consistency helps the model make better predictions.
- 2. Inspired by Language Modeling Theory
    - This idea comes from the intuition that language models are autoencoders of tokens — you encode a token, process it, and decode it. Why learn two embeddings when you can learn one and reuse it?

- 🧠 Analogy

- Think of it like a dictionary and its reverse lookup.
    - If "dog" maps to vector X in the encoder (embedding),

    - Then it's natural to reuse that mapping to decode X back to "dog".

## 📌 When not to share weights

- You might avoid weight sharing if:

    - You want the input and output spaces to diverge (e.g. translation from English to Hindi).

    - You're experimenting with more expressive output layers (e.g. adapter layers, soft prompts).

    - You use subword or byte-level tokenization and want distinct handling.

In [31]:
model = GPT(GPTConfig())
model.eval()
model = model.to(device)

In [32]:
## Optimizer 
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)
for i in range(50):
    x,y = train_loader.next_batch()
    x,y = x.to(device),y.to(device)
    t0  = time.time()
    optimizer.zero_grad()
    logits,loss = model(x,y)
    loss.backward()
    optimizer.step()
    t1  = time.time()
    dt = (t1 - t0) * 1000  # convert seconds to milliseconds
    print(f"Step : {i} loss: {loss.item():.4f}, dt: {dt:.4f} ms")

Step : 0 loss: 11.0446, dt: 33.1645 ms
Step : 1 loss: 9.4457, dt: 27.7996 ms
Step : 2 loss: 8.9811, dt: 36.3009 ms
Step : 3 loss: 9.0710, dt: 35.7182 ms
Step : 4 loss: 9.3238, dt: 38.2872 ms
Step : 5 loss: 8.3456, dt: 28.3477 ms
Step : 6 loss: 8.3575, dt: 28.0712 ms
Step : 7 loss: 7.9694, dt: 26.2806 ms
Step : 8 loss: 8.2492, dt: 28.9197 ms
Step : 9 loss: 8.6285, dt: 36.4096 ms
Step : 10 loss: 7.8400, dt: 27.4148 ms
Step : 11 loss: 7.7882, dt: 29.7611 ms
Step : 12 loss: 7.7437, dt: 20.9949 ms
Step : 13 loss: 7.5913, dt: 26.0835 ms
Step : 14 loss: 7.9870, dt: 27.7150 ms
Step : 15 loss: 6.8069, dt: 28.6427 ms
Step : 16 loss: 7.4417, dt: 26.6011 ms
Step : 17 loss: 7.3576, dt: 27.2703 ms
Step : 18 loss: 8.1371, dt: 29.8193 ms
Step : 19 loss: 8.5264, dt: 29.4244 ms
Step : 20 loss: 7.6777, dt: 27.4553 ms
Step : 21 loss: 8.0129, dt: 26.5503 ms
Step : 22 loss: 7.5895, dt: 22.4423 ms
Step : 23 loss: 7.3889, dt: 27.3418 ms
Step : 24 loss: 7.3109, dt: 26.4037 ms
Step : 25 loss: 7.5547, dt: 34.603